In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import mutual_info_score

In [3]:
url = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv"
df = pd.read_csv(url)
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [4]:
print("Missing values before preprocessing:")
print(df.isnull().sum())

Missing values before preprocessing:
lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64


```
Data preparation
Check if the missing values are presented in the features.
If there are missing values:
For categorical features, replace them with 'NA'
For numerical features, replace with with 0.0
```

In [5]:
categorical_cols = df.select_dtypes(include=["object"]).columns
numerical_cols = df.select_dtypes(exclude=["object"]).columns

df[categorical_cols] = df[categorical_cols].fillna("NA")
df[numerical_cols] = df[numerical_cols].fillna(0.0)

C:\Users\Admin3\AppData\Local\Temp\ipykernel_12552\2651458167.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df.select_dtypes(include=["object"]).columns


```
Question 1
What is the most frequent observation (mode) for the column industry?

NA
technology
healthcare
retail
```

In [7]:
print("\n1. Mode for industry")
print(df["industry"].mode().tolist())


1. Mode for industry
['technology']


```
Question 2
Create the correlation matrix for the numerical features of your dataset. In a correlation matrix, you compute the correlation coefficient between every pair of features.

What are the two features that have the biggest correlation?

interaction_count and lead_score
number_of_courses_viewed and lead_score
number_of_courses_viewed and interaction_count
annual_income and interaction_count
Only consider the pairs above when answering this question.
```

In [13]:
corr = df[numerical_cols].corr()

pairs = [
    ("interaction_count", "lead_score"),
    ("number_of_courses_viewed", "lead_score"),
    ("number_of_courses_viewed", "interaction_count"),
    ("annual_income", "interaction_count"),
]
print("\n2. Biggest correlation")
for a, b in pairs:
    print(f"{a} and {b}: {corr.loc[a, b]:.6f}")


2. Biggest correlation
interaction_count and lead_score: 0.915746
number_of_courses_viewed and lead_score: 0.757204
number_of_courses_viewed and interaction_count: 0.721609
annual_income and interaction_count: 0.122842


In [12]:
# Split the data with these exact calls:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)
# Make sure that the target value converted is not in your dataframe.
y_train = df_train["converted"].values
y_val = df_val["converted"].values

X_train = df_train.drop(columns=["converted"])
X_val = df_val.drop(columns=["converted"])


```
Question 3
Calculate the mutual information score between converted and other categorical variables in the dataset. Use the training set only.
Round the scores to 2 decimals using round(score, 2).
Which of these variables has the biggest mutual information score?

industry
location
lead_source
employment_status
```

In [14]:
categorical_features = X_train.select_dtypes(
    include=["object"]
).columns

mi_scores = {}
for col in categorical_features:
    mi_scores[col] = mutual_info_score(
        y_train, X_train[col]
    )

mi_scores = dict(
    sorted(mi_scores.items(), key=lambda item: item[1], reverse=True)
)

print("\n3. Biggest MI")
for feature, score in mi_scores.items():
    print(feature, round(score, 2))


C:\Users\Admin3\AppData\Local\Temp\ipykernel_12552\4267234611.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X_train.select_dtypes(



3. Biggest MI
lead_source 0.03
employment_status 0.02
industry 0.0
location 0.0


```
Question 4
Now let's train a logistic regression.
Remember that we have several categorical variables in the dataset. Include them using one-hot encoding.
Fit the model on the training dataset.
To make sure the results are reproducible across different versions of Scikit-Learn, fit the model with these parameters:
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
Calculate the accuracy on the validation dataset and round it to 2 decimal digits.
What accuracy did you get?

0.55
0.65
0.75
0.85
```

In [26]:
y_train = df_train["converted"].copy()
y_val = df_val["converted"].copy()

df_train = df_train.drop(columns=["converted"])
df_val = df_val.drop(columns=["converted"])

In [28]:
dv = DictVectorizer(sparse=False)

X_train = dv.fit_transform(df_train.to_dict(orient="records"))
X_val = dv.transform(df_val.to_dict(orient="records"))

model = LogisticRegression(
    solver="liblinear",
    C=1.0,
    max_iter=1000,
    random_state=42
)

model.fit(X_train, y_train)
y_pred = model.predict(X_val)

print("\n4. Accuracy")
print("Rounded accuracy:", round(accuracy_score(y_val, y_pred), 2))



4. Accuracy
Rounded accuracy: 0.65


```
Question 5
Let's find the least useful feature using the feature elimination technique.
Train a model using the same features and parameters as in Q4 (without rounding).
Now exclude each feature from this set and train a model without it. Record the accuracy for each model.
For each feature, calculate the difference between the original accuracy and the accuracy without the feature.
Which of following feature has the smallest difference?

'lead_source'
'number_of_courses_viewed'
'interaction_count'
Note: The difference doesn't have to be positive.
```

In [34]:
print("\n5. Feature selection")
differences = {}

for feature in [
    "lead_source",
    "number_of_courses_viewed",
    "interaction_count"
]:
    reduced_train = df_train.drop(columns=[feature])
    reduced_val = df_val.drop(columns=[feature])

    reduced_accuracy, _ = fit_model(
        reduced_train, reduced_val, y_train, C=1.0
    )

    differences[feature] = abs(
        original_accuracy - reduced_accuracy
    )

print(differences)
print(min(differences, key=differences.get))




5. Feature selection
{'lead_source': 0.358, 'number_of_courses_viewed': 0.357, 'interaction_count': 0.399}
number_of_courses_viewed


```
Question 6
Now let's train a regularized logistic regression.
Let's try the following values of the parameter C: [0.000001, 0.00001, 0.0001, 0.001].
Train models using all the features as in Q4.
Calculate the accuracy on the validation dataset and round it to 3 decimal digits.
Which of these C leads to the best accuracy on the validation set?

0.000001
0.00001
0.0001
0.001
Note: If there are multiple options, select the smallest C.
```

In [35]:
print("\n6. Parameter tuning")
scores = {}

for C in [0.000001, 0.00001, 0.0001, 0.001]:
    accuracy, _ = fit_model(
        df_train, df_val, y_train, C=C
    )
    scores[C] = accuracy
    print(f"C={C}: accuracy={accuracy:.3f}")

best_C = max(scores, key=lambda c: (scores[c], -c))
print(best_C)


6. Parameter tuning
C=1e-06: accuracy=0.598
C=1e-05: accuracy=0.598
C=0.0001: accuracy=0.613
C=0.001: accuracy=0.645
0.001
